# Pipeline Bronze → Silver — TLC Trip Record Data

Este notebook ejecuta la transformación de la capa **Bronze** (datos crudos)
a la capa **Silver** (datos limpios, unificados y enriquecidos) dentro de la
arquitectura Medallion del proyecto.

**Lo que hace:**
- Lee los 4 tipos de vehículo (`yellow`, `green`, `fhv`, `fhvhv`) desde Bronze
- Unifica los esquemas a una estructura común (ver `src/silver/schemas.py`)
- Aplica las reglas de limpieza definidas en el EDA (ver `docs/decisiones_limpieza.md`)
- Agrega columnas calculadas (`duracion_min`, `hora`, `dia_semana`)
- Escribe a Silver particionado por `tipo_vehiculo / anio / mes`

**Diseño:** el procesamiento se hace archivo por archivo (no unión completa
en memoria) debido a la limitación de RAM del entorno (8GB). Cada función
incluye reintento automático ante fallos de memoria de Spark.

La lógica vive en `src/`, este notebook solo orquesta la ejecución.

## 1. Imports y verificación de configuración

Se cargan las funciones desde `src/` y se valida que la configuración
del proyecto (años, tipos de vehículo) sea la esperada antes de ejecutar
cualquier transformación.

In [1]:
import sys
sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session, ejecutar_con_reintento
from silver.transform import PROCESADORES
from config import ANIOS, MESES, TIPOS_VEHICULO

print("Imports OK")
print("Tipos:", TIPOS_VEHICULO)
print("Años:", ANIOS)

Imports OK
Tipos: ['yellow', 'green', 'fhv', 'fhvhv']
Años: [2023, 2024, 2025]


## 2. Ejecución del pipeline Bronze → Silver

Se itera por cada tipo de vehículo, año y mes. Por cada combinación:

1. Se verifica si ya fue procesado (`skip` automático, permite reanudar
   tras un fallo sin reprocesar lo ya hecho)
2. Se lee el archivo Bronze correspondiente
3. Se renombran columnas al esquema Silver unificado
4. Se aplican las reglas de limpieza específicas del tipo de vehículo
5. Se valida que el `pickup_datetime` corresponda al mes/año del archivo
   (corrige registros con fechas corruptas detectados en el EDA)
6. Se escribe a disco y se registra auditoría en `data/logs/auditoria_silver.csv`

Si Spark falla por memoria, `ejecutar_con_reintento()` reinicia la sesión
automáticamente y reintenta hasta 3 veces antes de marcar el mes como fallido.

In [ ]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session, ejecutar_con_reintento
from silver.transform import PROCESADORES
from config import ANIOS, MESES, TIPOS_VEHICULO

spark = crear_spark_session()

for tipo in TIPOS_VEHICULO:
    print(f"\n{'='*50}")
    print(f"  PROCESANDO: {tipo.upper()}")
    print(f"{'='*50}")
    
    for anio in ANIOS:
        for mes in MESES:
            spark, exito = ejecutar_con_reintento(
                spark, PROCESADORES[tipo], anio, mes
            )
            if not exito:
                print(f"  [FALLO DEFINITIVO] {tipo} {anio}-{mes:02d}")

print("\n\nSilver completo!")
spark.stop()

2026-07-03 19:36:51,547 [INFO] SparkSession creada: 3.5.0



  PROCESANDO: YELLOW


2026-07-03 19:37:11,275 [INFO] [OK] yellow 2023-01: 3,066,766 -> 2,989,282 filas
2026-07-03 19:37:19,442 [INFO] [OK] yellow 2023-02: 2,913,955 -> 2,841,526 filas
2026-07-03 19:37:28,248 [INFO] [OK] yellow 2023-03: 3,403,766 -> 3,317,032 filas
2026-07-03 19:37:37,119 [INFO] [OK] yellow 2023-04: 3,288,250 -> 3,209,566 filas
2026-07-03 19:37:46,439 [INFO] [OK] yellow 2023-05: 3,513,649 -> 3,426,929 filas
2026-07-03 19:37:54,903 [INFO] [OK] yellow 2023-06: 3,307,234 -> 3,221,181 filas
2026-07-03 19:38:02,236 [INFO] [OK] yellow 2023-07: 2,907,108 -> 2,819,724 filas
2026-07-03 19:38:07,271 [INFO] [OK] yellow 2023-08: 2,824,209 -> 2,729,767 filas
2026-07-03 19:38:12,525 [INFO] [OK] yellow 2023-09: 2,846,722 -> 2,713,343 filas
2026-07-03 19:38:17,629 [INFO] [OK] yellow 2023-10: 3,522,285 -> 3,357,405 filas
2026-07-03 19:38:22,368 [INFO] [OK] yellow 2023-11: 3,339,715 -> 3,193,557 filas
2026-07-03 19:38:26,860 [INFO] [OK] yellow 2023-12: 3,376,567 -> 3,254,505 filas
2026-07-03 19:38:31,051 [INF

## 3. Verificación de resultados

Confirmamos que el Silver quedó completo y sin inconsistencias de fecha
antes de pasar a la capa Gold.

In [7]:
import os
from config import SILVER_PATH

total_meses = 0
for root, dirs, files in os.walk(SILVER_PATH):
    if "mes=" in root:
        total_meses += 1

print(f"Total de particiones mes encontradas: {total_meses}")

Total de particiones mes encontradas: 144


In [8]:
from spark_utils import crear_spark_session
spark = crear_spark_session()

2026-06-30 08:55:52,950 [INFO] SparkSession creada: 3.5.0


In [9]:
df_silver = spark.read.parquet("/home/jovyan/data/silver/trips/")
df_silver.createOrReplaceTempView("trips")
spark.sql("SELECT tipo_vehiculo, COUNT(*) FROM trips GROUP BY tipo_vehiculo").show()

+-------------+---------+
|tipo_vehiculo| count(1)|
+-------------+---------+
|       yellow|120733122|
|        fhvhv|713669341|
|          fhv| 11218792|
|        green|  1903467|
+-------------+---------+



In [7]:
import pandas as pd
from config import LOGS_PATH

aud = pd.read_csv(LOGS_PATH / "auditoria_silver.csv")
print(aud[aud["tipo_vehiculo"].isin(["yellow", "green"])].to_string())

2026-07-03 18:05:47,622 [INFO] NumExpr defaulting to 4 threads.


   tipo_vehiculo  anio  mes  filas_entrada  filas_salida  pct_retenido
0         yellow  2023    1        3066766       2989282         97.47
1         yellow  2023    1        3066766       2989282         97.47
2         yellow  2023    2        2913955       2841526         97.51
3         yellow  2023    3        3403766       3317032         97.45
4         yellow  2023    4        3288250       3209566         97.61
5         yellow  2023    5        3513649       3426929         97.53
6         yellow  2023    6        3307234       3221181         97.40
7         yellow  2023    7        2907108       2819724         96.99
8         yellow  2023    8        2824209       2729767         96.66
9         yellow  2023    9        2846722       2713343         95.31
10        yellow  2023   10        3522285       3357405         95.32
11        yellow  2023   11        3339715       3193557         95.62
12        yellow  2023   12        3376567       3254505         96.39
13    